# Python prompting on the situational test cases

**Research question.** Can an LLM turn a short, under-specified description of a real
situation into a *useful formal world model* — here, an executable Python subclass of this
repository's `AbstractPOSG` (partially observed stochastic game) interface?

This notebook is the Python-track counterpart to the JSON track
(`prompts_library/zero_shot_prompt.md`, evaluated by Lane 1 / Lane 2 in
`notebooks/Copy_of_Test_Cases (1).ipynb`). It runs on the **same 11 situational cases**
(`data/Test_Cases/02_situational/`) with the **same scenario text**, so the two tracks stay
comparable.

### Three levels of correctness, kept separate

| Level | Question | How it is answered here |
|---|---|---|
| 1. Python | Does it parse, import, instantiate, and subclass `AbstractPOSG`? | `ast.parse` + a subprocess probe |
| 2. Mechanical POSG | Are the distributions, action symbols, state tuples, observation profiles and reward vectors well formed? | the repository's `AbstractPOSG.validate()` **plus** an added exhaustive sweep over joint actions |
| 3. Semantic / world-model | Are the right humans, options, uncertainties and observations in there? | **not automated** — structured extracts + the repository's own expect/avoid checklists, prepared for human review |

**valid Python ≠ valid POSG ≠ good world model.** A class can pass every mechanical check
while inventing a stakeholder, collapsing an uncertainty, or handing a player information
they could not have. Lane 3 of the evaluation methodology (expect/avoid ingredient coverage)
is *not implemented anywhere in this repository*, and this notebook does not invent a
substitute: section J emits review sheets and mechanical flags, and marks the semantic
verdict as `pending_human_review`.

### What the pipeline does

```
situation (repo markdown)
  -> prompt (interface excerpt + evidence discipline)
    -> LLM generation
      -> extract code, write module
        -> subprocess: import, instantiate, validate(), sweep
          -> [optional] one repair round with the exact validator errors
            -> results table + per-case review sheet
```

### How to run

1. **Colab:** `Runtime -> Run all`. The first cell clones the repository and installs
   `anthropic`. Put your key in Colab *Secrets* as `ANTHROPIC_API_KEY` (notebook access on).
2. **Locally:** `export ANTHROPIC_API_KEY=...` and run from the repository checkout.
3. **Without a key:** every cell still runs. The notebook drops into `dry_run` mode, which
   substitutes a hard-coded stub "generation" so the harness itself can be exercised and
   tested. Stub rows are labelled `dry-run-stub` and are **not** experimental results.

Nothing outside the run directory (printed by the setup cell) is written.


## A. Setup


In [ ]:
# --- environment -------------------------------------------------------------------
import subprocess, sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/AmnaJamalKhattak/world-model-suite.git"
REPO_NAME = "world-model-suite"

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic", "pandas"], check=False)
    if not Path(REPO_NAME).exists():
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)


def find_repo_root(start: Path) -> Path:
    """Walk up (then down into a fresh clone) looking for the test-case directory."""
    marker = Path("data") / "Test_Cases" / "02_situational"
    for candidate in [start, *start.parents]:
        if (candidate / marker).is_dir():
            return candidate
        if (candidate / REPO_NAME / marker).is_dir():
            return candidate / REPO_NAME
    raise FileNotFoundError(f"could not locate {marker} from {start}")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
print("repository root:", REPO_ROOT)


In [ ]:
# --- imports and configuration ------------------------------------------------------
import ast, dataclasses, datetime as dt, hashlib, itertools, json, os, re, textwrap, types
from dataclasses import dataclass, field

import pandas as pd
from IPython.display import Markdown, display

POSG_NOTEBOOK = REPO_ROOT / "notebooks" / "POSG_World_Model_Implementation.ipynb"
SITUATIONAL_DIR = REPO_ROOT / "data" / "Test_Cases" / "02_situational"
OUT_ROOT = REPO_ROOT / "experiments" / "python_posg_situational"


def api_key() -> str:
    key = os.environ.get("ANTHROPIC_API_KEY", "")
    if not key and IN_COLAB:
        try:
            from google.colab import userdata
            key = userdata.get("ANTHROPIC_API_KEY") or ""
        except Exception:
            key = ""
    return key


@dataclass
class Config:
    """Everything the experiment depends on, in one place and saved with the run."""

    # generation
    model: str = "claude-opus-5"
    effort: str = "high"              # low | medium | high | xhigh | max
    max_tokens: int = 16000
    adaptive_thinking: bool = True
    # A refused request would otherwise be silently re-served by a *different* model, which
    # would mix two models under one label in the results table. Refusals are recorded as an
    # outcome instead. Flip to True only if you also read `served_by` in the results.
    use_refusal_fallback: bool = False

    # experiment shape
    n_generations: int = 1            # independent generations per case
    max_repairs: int = 1              # validator-feedback repair rounds per generation
    case_filter: tuple = ()           # e.g. ("S.5", "S.10"); empty = all cases

    # execution / exploration caps (a cap that is hit is itself a finding)
    probe_timeout_s: int = 60
    caps: dict = field(default_factory=lambda: dict(
        max_states=400, max_profiles=200, max_depth=12, max_samples=8, max_errors=25))

    dry_run: bool = False             # set automatically when no API key is available
    seed: int = 0


CFG = Config()
if not api_key():
    CFG.dry_run = True
    print("!! no ANTHROPIC_API_KEY found -> dry_run mode: the harness runs against a stub,\n"
          "   and the resulting rows are NOT experimental results.")

RUN_ID = f"{dt.datetime.now():%Y%m%d-%H%M%S}_{'dry-run' if CFG.dry_run else CFG.model}"
RUN_DIR = OUT_ROOT / RUN_ID
for sub in ("prompts", "raw", "generated", "probe", "review"):
    (RUN_DIR / sub).mkdir(parents=True, exist_ok=True)
(RUN_DIR / "config.json").write_text(json.dumps(dataclasses.asdict(CFG), indent=2, default=str))
print("run directory:", RUN_DIR)


## B. The POSG interface — existing repository code

`notebooks/POSG_World_Model_Implementation.ipynb` holds the authoritative interface
(`AbstractPOSG` plus the `freeze` / `unfreeze` state helpers) and two reference
implementations. It is a notebook, not an importable module, so the cells that define the
interface are extracted and executed here — the interface is **not** copied into this
notebook, so it cannot drift.

The PettingZoo wrapper and the demo cells are left behind: the interface itself needs
neither `pettingzoo` nor `gymnasium`, and a bare Colab has neither installed.

Two properties of the interface drive the whole prompt, and both come from reading
`validate()`:

* **every player acts in every non-terminal state** (simultaneous move) — a player with
  nothing to do still needs a no-op action, or validation fails;
* `initial_state` is a **single state**, not a distribution — so an unknown fact has to be
  carried as a state variable and resolved inside a transition.


In [ ]:
def extract_posg_abc_source(nb_path: Path) -> str:
    """Take only the cells the interface needs out of the reference notebook."""
    nb = json.loads(nb_path.read_text())
    wanted, seen_abc = [], False
    for cell in nb["cells"]:
        if cell["cell_type"] != "code":
            continue
        src = "".join(cell["source"])
        if "from abc import" in src or "def freeze(" in src or "class AbstractPOSG" in src:
            wanted.append(src)
            seen_abc = seen_abc or "class AbstractPOSG" in src
    if not seen_abc:
        raise RuntimeError(f"no AbstractPOSG definition found in {nb_path}")
    src = "\n\n".join(wanted)
    return "\n".join(l for l in src.splitlines()
                     if "pettingzoo" not in l and "gymnasium" not in l)


def load_module(src: str, name: str) -> types.ModuleType:
    mod = types.ModuleType(name)
    mod.__file__ = f"<{name}>"
    exec(compile(src, f"<{name}>", "exec"), mod.__dict__)
    return mod


ABC_SOURCE = extract_posg_abc_source(POSG_NOTEBOOK)
ABC_PATH = RUN_DIR / "posg_abc.py"
ABC_PATH.write_text("# extracted verbatim from notebooks/POSG_World_Model_Implementation.ipynb\n"
                    "# (interface cells only) — do not edit here, edit the reference notebook.\n\n"
                    + ABC_SOURCE)
posg_abc = load_module(ABC_SOURCE, "posg_abc_inline")
AbstractPOSG = posg_abc.AbstractPOSG

print("interface loaded from:", POSG_NOTEBOOK.relative_to(REPO_ROOT))
print("abstract surface:", sorted(AbstractPOSG.__abstractmethods__))
print("provided helpers :", ["freeze", "unfreeze", "validate", "schema"])


In [ ]:
def interface_excerpt(src: str) -> str:
    """Signatures + docstrings only: what an implementer needs, without the machinery.

    Derived from the reference notebook with `ast`, so it tracks changes to the interface
    instead of being a hand-copied snapshot. `validate` and `schema` keep their signatures
    and get a one-line summary of what they do, so the model knows what is checked.
    """
    provided = {
        "validate": ("Provided. Returns a list of error strings. It checks: metadata is non-empty; "
                     "the initial state is hashable; every player has at least one legal action in "
                     "every reachable non-terminal state; every legal action is in the action "
                     "catalogue; kernel probabilities lie in [0, 1] and sum to 1; the observation "
                     "profile has exactly one entry per declared player."),
        "schema": "Provided. Renders the declared metadata as readable text.",
    }
    tree, out = ast.parse(src), []
    for node in tree.body:
        if isinstance(node, ast.FunctionDef) and node.name in ("freeze", "unfreeze"):
            out.append(ast.unparse(node))
        elif isinstance(node, ast.ClassDef) and node.name == "AbstractPOSG":
            body = []
            for item in node.body:
                if isinstance(item, ast.Expr) and isinstance(item.value, ast.Constant):
                    body.append(item)
                elif isinstance(item, ast.FunctionDef):
                    decorators = " ".join(ast.unparse(d) for d in item.decorator_list)
                    if "abstractmethod" in decorators:
                        item.body = [ast.Expr(ast.Constant(ast.get_docstring(item) or "..."))]
                        body.append(item)
                    elif item.name in provided:
                        item.decorator_list = []
                        item.body = [ast.Expr(ast.Constant(provided[item.name]))]
                        body.append(item)
            node.body = body
            out.append(ast.unparse(node))
    return "\n\n".join(out)


INTERFACE_EXCERPT = interface_excerpt(ABC_SOURCE)
print(f"excerpt: {len(INTERFACE_EXCERPT)} chars (~{len(INTERFACE_EXCERPT)//4} tokens) "
      f"of {len(ABC_SOURCE)} in the full interface cell\n")
print(INTERFACE_EXCERPT[:900] + "\n...")


## C. Situational test cases — existing repository data

The eleven situational cases live in `data/Test_Cases/02_situational/*.md`, one file per
case, each with an `### LLM Input` section (the scenario handed to the model, **used
verbatim** so the Python and JSON tracks stay comparable) and a `### Judging Criteria`
section split into *should include* and *should NOT include* bullets.

Those two bullet lists are the repository's expect/avoid annotation — the closest thing it
has to the MUST / FORBID ingredient methodology — and they are carried through to the
review sheets in section J. The toy, game and cultural tracks are deliberately **not**
loaded here.


In [ ]:
_HEADING = re.compile(r"^##\s*Test Case\s+(S\.\d+)\s*:\s*(.+?)\s*$", re.M)
_SECTION = re.compile(r"^###\s+(.*?)\s*$", re.M)
_NOT_LINE = re.compile(r"should\s*\**\s*NOT\s+include", re.I)
_BULLET = re.compile(r"^\s*[-*]\s+(.*\S)\s*$")


def _clean(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip().rstrip(";").strip()


def parse_case(path: Path) -> dict:
    md = path.read_text()
    head = _HEADING.search(md)
    if not head:
        raise ValueError(f"{path.name}: no '## Test Case S.N: ...' heading")

    marks = list(_SECTION.finditer(md))
    sections = {}
    for i, m in enumerate(marks):
        end = marks[i + 1].start() if i + 1 < len(marks) else len(md)
        sections[m.group(1).strip().lower()] = md[m.end():end].strip()

    lines = sections.get("judging criteria", "").splitlines()
    # the "...should NOT include" line is the divider; its markdown emphasis varies by file
    split_at = next((i for i, l in enumerate(lines) if _NOT_LINE.search(l)), len(lines))
    bullets = lambda block: [_clean(m.group(1)) for line in block
                             for m in [_BULLET.match(line)] if m]

    case = {
        "case_id": head.group(1),
        "title": head.group(2),
        "situation": _clean(sections.get("llm input", "")),
        "expect": bullets(lines[:split_at]),
        "avoid": bullets(lines[split_at:]),
        "source_file": str(path.relative_to(REPO_ROOT)),
    }
    if not case["situation"]:
        raise ValueError(f"{path.name}: empty '### LLM Input' section")
    return case


CASES = sorted((parse_case(p) for p in SITUATIONAL_DIR.glob("*.md")),
               key=lambda c: int(c["case_id"].split(".")[1]))
if CFG.case_filter:
    CASES = [c for c in CASES if c["case_id"] in CFG.case_filter]

pd.DataFrame([{"case": c["case_id"], "title": c["title"],
               "situation_words": len(c["situation"].split()),
               "expect": len(c["expect"]), "avoid": len(c["avoid"])} for c in CASES])


## D. The prompt — added by this notebook

The prompt has three parts: the interface excerpt from section B, an **evidence
discipline**, and the rules the validator actually enforces.

### Evidence discipline: given / inferred / uncertain / invented

A blanket "do not assume anything beyond the prompt" would make world modelling
impossible — a traffic model needs to know that cars brake. The prompt distinguishes four
kinds of content instead, matching the *given / inferred / invented* discipline already
used in the repository's Lane-1 docstring and the `Evidence` section of
`docs/representation_science/representation_canvas.md`:

| Kind | Instruction |
|---|---|
| stated in the situation | treat as true |
| ordinary background knowledge | may be used where the situation needs it |
| important but unknown | keep **both** possibilities live; do not pick one |
| unsupported specifics | do not invent (ages, names, exact speeds, backstories) |

### Numerical probabilities — an unresolved question, flagged rather than settled

Lane 1 only checks that probabilities *normalise*; nothing in the repository says what to
do when the LLM writes `0.2` for a chance the scenario never quantified, and
`docs/representation_science/open_questions.md` still asks "how should a metric distinguish
unsupported invention from useful commonsense inference?". This notebook therefore does
**not** decide the question. It adopts one conservative, auditable convention and measures
how well it holds:

* a probability that the situation does not fix must be a **module-level named constant**
  carrying an `# UNSUPPORTED:` comment, and should be an uninformative value (`0.5`, or
  `1/n`) rather than a specific-looking guess;
* section G audits the generated source for bare numeric literals inside `transition_kernel`
  and reports them next to the declared constants.

That is a notebook-level convention, **not** ratified project methodology — see section K.

### Rewards

The interface makes `rewards` abstract, so it must be implemented; the project's JSON
prompt says the model supplies "dynamics only" because the downstream power analysis
derives value itself. The prompt reconciles the two by asking for `0.0` for every player
unless the situation states an explicit payoff. Also an open question (section K).


In [ ]:
SYSTEM_PROMPT = """You are a world-model builder. You turn a natural-language SITUATION into a formal
world model: a Python class implementing the POSG (partially observed stochastic game)
interface below.

The model is consumed by a separate analysis that measures how much power the human players
retain to pursue a wide range of possible goals. Your job is the *dynamics only* — who acts,
what they can do, what may follow, and who observes what. You supply no goals, preferences or
judgements about what is good.

You are given only the situation. The players, their actions, the state variables and the
transitions are not listed for you: inferring them from ordinary world knowledge is the task.

# The interface (already imported; do not redefine or re-import it)

```python
{interface}
```

`freeze(dict) -> frozenset` and `unfreeze(frozenset) -> dict` are available, as are
`Any, Dict, List, Optional, Tuple, FrozenSet` from `typing`.

# What the validator enforces

* All players move **simultaneously**: `legal_actions(state, player)` must return at least one
  action for **every** player in **every** non-terminal state. Give a player who is doing
  nothing an explicit no-op (`wait`, `do_nothing`) rather than an empty list.
* `legal_actions` returns `[]` exactly when `is_terminal(state)` is True.
* Every action returned anywhere must be a key of `action_catalogue`.
* Every state is `freeze({{...}})` over exactly the declared `state_variables` — the same keys
  in every state, terminal states included. Values must be hashable (str, int, bool, None).
* `transition_kernel` returns `{{(next_state, observation_profile): probability}}` with
  probabilities > 0 summing to 1. `observation_profile` is `freeze({{player: observation}})`
  with exactly one entry per declared player.
* `rewards` returns exactly one float per declared player.
* `initial_state` is a single state, not a distribution.

# How to represent the things that matter

* **Uncertainty about a fact** (is the warning true? is anyone home?): carry it as a state
  variable, start it at `"unknown"`, and resolve it inside a transition — never let a player
  choose an action that depends on a value nobody has observed yet.
* **Partial observability**: the observation profile is where players differ. A player always
  observes their own action and what they could plainly see; they do not observe a latent
  variable that has not been revealed to them.
* **Information gathering** (ask, check, inspect, call, confirm) is an action like any other,
  and must be available *before* any irreversible action.
* **Rewards**: return 0.0 for every player unless the situation states an explicit payoff. The
  downstream analysis derives value itself.

# Evidence discipline

1. **Stated in the situation** — treat as true.
2. **Ordinary background knowledge** — use it where the situation needs it (cars brake,
   pedestrians can move, a delivery has a recipient, a collision may hurt someone).
3. **Important but unknown** — if the situation leaves a decision-relevant fact open, keep
   both possibilities live in the model. Do not pick one and call it settled.
4. **Unsupported specifics** — do not invent ages, names, identities, backstories, exact
   speeds, quantities or institutional rules that the situation does not imply.

Probabilities follow the same discipline. If the situation does not fix a probability, do not
write a specific-looking number inline: declare a module-level constant above the class with an
`# UNSUPPORTED:` comment naming what is unknown, and give it an uninformative value (0.5 for an
unknown binary fact, 1/n for n indistinguishable outcomes). For example:

```python
P_GUARDIAN_NEARBY = 0.5  # UNSUPPORTED: the situation does not say whether an adult is nearby
```

# Size

Keep the model small enough for a person to read and check: roughly 2-5 players, 3-6 state
variables, 2-5 actions per player per state, and two to four decision points before a terminal
state. A distinction earns its place only if it changes what someone can do or what may follow.
Prefer the smallest model that still shows the real interaction.

# Output

Work out the representation before you write anything — that thinking is not part of the
output. Then return **only** the Python code: optional module-level constants followed by one
class `SituationPOSG(AbstractPOSG)`. No prose, no markdown fences, no example usage, no
`if __name__ == "__main__"` block."""


REPAIR_PROMPT = """The class you produced for this situation fails mechanical validation.

Your previous code:

```python
{code}
```

Reported problems ({stage}):

```
{errors}
```

Return the complete corrected class again, applying the **smallest change that fixes the
reported problems**. Do not change the modelling decisions — the players, actions, state
variables and uncertainties should stay as they are unless a problem requires otherwise.
Same output rules: only the Python code, no prose and no fences."""


def build_messages(case: dict, previous_code: str = "", errors: str = "", stage: str = "") -> tuple:
    """Return (system_prompt, user_prompt) for a generation or a repair round."""
    system = SYSTEM_PROMPT.format(interface=INTERFACE_EXCERPT)
    user = f"SITUATION (test case {case['case_id']}):\n\n{case['situation']}"
    if previous_code:
        user += "\n\n" + REPAIR_PROMPT.format(code=previous_code, errors=errors[:4000], stage=stage)
    return system, user


In [ ]:
_sys, _usr = build_messages(CASES[0])
print(f"system prompt: {len(_sys)} chars (~{len(_sys)//4} tokens)")
print(f"user prompt  : {len(_usr)} chars (~{len(_usr)//4} tokens)\n")
print(_usr)
print("\n--- system prompt, first 1200 chars ---\n")
print(_sys[:1200] + " ...")


## E. Generation — added by this notebook

The repository has no model-calling helper (the earlier `llm_world_model_experiment/`
prototype templated its "generated" code rather than calling an API), so one is added here.
It is a single seam: `call_llm(system, user)` returns `(text, metadata)`, and swapping in a
different provider means replacing that one function.

Token counts, stop reason and the serving model are recorded per call. `stop_reason ==
"refusal"` is treated as a recorded outcome rather than an error, and no silent model
fallback is enabled (see `Config.use_refusal_fallback`).


In [ ]:
_client = None


def call_llm(system: str, user: str) -> tuple:
    """(text, meta). The single provider seam — replace this body to use another API."""
    global _client
    import anthropic

    if _client is None:
        _client = anthropic.Anthropic(api_key=api_key())

    kwargs = dict(model=CFG.model, max_tokens=CFG.max_tokens, system=system,
                  messages=[{"role": "user", "content": user}])
    if CFG.adaptive_thinking:
        kwargs["thinking"] = {"type": "adaptive"}
    if CFG.effort:
        kwargs["output_config"] = {"effort": CFG.effort}

    if CFG.use_refusal_fallback:
        kwargs["betas"] = ["server-side-fallback-2026-07-01"]
        kwargs["fallbacks"] = "default"
        response = _client.beta.messages.create(**kwargs)
    else:
        response = _client.messages.create(**kwargs)

    text = "".join(b.text for b in response.content if getattr(b, "type", "") == "text")
    meta = {
        "served_by": response.model,
        "stop_reason": response.stop_reason,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
        "cache_read_input_tokens": getattr(response.usage, "cache_read_input_tokens", None),
        "refusal": response.stop_reason == "refusal",
    }
    if meta["refusal"]:
        details = getattr(response, "stop_details", None)
        meta["refusal_category"] = getattr(details, "category", None) if details else None
    return text, meta


CODE_FENCE = re.compile(r"```(?:python|py)?\s*\n(.*?)```", re.S)


def extract_python(text: str) -> str:
    """Pull the implementation out of the response, fenced or not."""
    blocks = CODE_FENCE.findall(text or "")
    if blocks:
        return max(blocks, key=len).strip()
    if re.search(r"^\s*(class\s+\w+\s*\(|[A-Z_]+\s*=)", text or "", re.M):
        return (text or "").strip()
    return ""


In [ ]:
# --- dry-run stub: a fixture for testing the harness, never an experimental result --------
STUB_CODE = '''\
P_HARM_IF_UNCHECKED = 0.5  # UNSUPPORTED: the situation does not quantify this risk

class SituationPOSG(AbstractPOSG):
    """DRY-RUN STUB. Deliberately thin: one robot, one human, one uncertainty."""

    @property
    def players(self):
        return {"robot": {"description": "The AI agent in the situation."},
                "human": {"description": "The one human this stub represents."}}

    @property
    def state_variables(self):
        return {"phase": {"type": "str", "description": "start | acted | done"},
                "hazard": {"type": "str", "description": "unknown | real | absent"}}

    @property
    def action_catalogue(self):
        return {"check": {"description": "Gather information before acting."},
                "act": {"description": "Take the task action."},
                "wait": {"description": "Do nothing this step."}}

    @property
    def initial_state(self):
        return freeze({"phase": "start", "hazard": "unknown"})

    def legal_actions(self, state, player):
        if self.is_terminal(state):
            return []
        return ["check", "act", "wait"] if player == "robot" else ["wait"]

    def transition_kernel(self, state, actions):
        s = unfreeze(state)
        out = {}
        if actions["robot"] == "check" and s["hazard"] == "unknown":
            for value, p in (("real", P_HARM_IF_UNCHECKED), ("absent", 1.0 - P_HARM_IF_UNCHECKED)):
                nxt = freeze({"phase": "acted", "hazard": value})
                obs = freeze({"robot": f"observed_{value}", "human": "robot_checked"})
                out[(nxt, obs)] = p
            return out
        phase = "done" if actions["robot"] == "act" else "acted"
        nxt = freeze({"phase": phase, "hazard": s["hazard"]})
        obs = freeze({"robot": f"did_{actions['robot']}", "human": "nothing_new"})
        return {(nxt, obs): 1.0}

    def rewards(self, state, actions, next_state):
        return {"robot": 0.0, "human": 0.0}

    def is_terminal(self, state):
        return unfreeze(state)["phase"] == "done"
'''

# the same stub with a defect, so the repair loop has something to repair in dry-run mode
STUB_BROKEN = STUB_CODE.replace("1.0 - P_HARM_IF_UNCHECKED", "0.3")
DRY_RUN_BROKEN_CASES = {"S.5"}


def stub_generate(case: dict, is_repair: bool) -> tuple:
    code = STUB_BROKEN if (case["case_id"] in DRY_RUN_BROKEN_CASES and not is_repair) else STUB_CODE
    meta = {"served_by": "dry-run-stub", "stop_reason": "end_turn", "input_tokens": 0,
            "output_tokens": 0, "cache_read_input_tokens": None, "refusal": False}
    return f"```python\n{code}```", meta


def generate(case: dict, previous_code: str = "", errors: str = "", stage: str = "") -> tuple:
    """(raw_text, code, meta, prompts) for one generation or repair round."""
    system, user = build_messages(case, previous_code, errors, stage)
    if CFG.dry_run:
        raw, meta = stub_generate(case, bool(previous_code))
    else:
        try:
            raw, meta = call_llm(system, user)
        except Exception as exc:                      # API failure is a recorded outcome
            raw, meta = "", {"served_by": None, "stop_reason": "api_error", "input_tokens": 0,
                             "output_tokens": 0, "cache_read_input_tokens": None,
                             "refusal": False, "api_error": f"{type(exc).__name__}: {exc}"}
    return raw, extract_python(raw), meta, {"system": system, "user": user}


## F. Execution and mechanical validation

Generated code is executed in a **subprocess** with a timeout, so a runaway state space or
an infinite loop cannot take the notebook down (the same isolation the earlier
`llm_world_model_experiment/run_experiment.py` used). The probe reports, in order:

1. **import** — module loads (syntax is checked in-process first, with `ast.parse`);
2. **class** — the module defines a subclass of `AbstractPOSG`;
3. **instantiate** — no abstract method left unimplemented;
4. **validate** — the repository's own `AbstractPOSG.validate()`, unchanged;
5. **sweep** — added here.

The sweep exists because `validate()` follows only the *first* legal action profile out of
each state, and does not look at rewards or at state-variable agreement. The sweep walks the
reachable graph over the **full joint-action product** (within caps) and additionally checks
that every state carries exactly the declared state variables, that reward vectors have one
numeric entry per player, and that observation profiles cover the players — while collecting
the material section J needs (probabilities used, observations per player, terminal states).

Caps (`Config.caps`) bound the walk. **A cap that is hit is reported, not hidden**: it means
the model is too large to verify exhaustively, which is itself a finding.


In [ ]:
PROBE_SOURCE = r'''
"""Execute one generated POSG module in a fresh interpreter and report on it.

Usage:  python probe.py <posg_abc.py> <module.py> <caps.json>
Always prints a single JSON object on stdout, whatever goes wrong.
"""
from __future__ import annotations

import importlib.util
import io
import itertools
import json
import math
import sys
import traceback


def _jsonify(value):
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    if isinstance(value, frozenset):
        try:
            return {str(k): _jsonify(v) for k, v in dict(value).items()}
        except Exception:
            return sorted(repr(v) for v in value)
    if isinstance(value, dict):
        return {str(k): _jsonify(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [_jsonify(v) for v in value]
    return repr(value)


def _err(exc):
    return {"type": type(exc).__name__, "message": str(exc)[:2000],
            "traceback": "".join(traceback.format_exception(
                type(exc), exc, exc.__traceback__))[-4000:]}


def _load(path, name):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


def sweep(game, caps):
    """Walk the reachable state graph over all joint actions, within caps."""
    errors, warnings, caps_hit = [], [], []
    players = list(game.players.keys())
    declared_vars = set(game.state_variables.keys())
    catalogue = set(game.action_catalogue.keys())

    probabilities, actions_used = set(), set()
    observations = {p: set() for p in players}
    terminals, reward_samples, n_entries = [], [], 0
    seen, frontier, max_depth = set(), [(game.initial_state, 0)], 0

    def note(msg):
        if msg not in errors and len(errors) < caps["max_errors"]:
            errors.append(msg)

    while frontier:
        state, depth = frontier.pop(0)
        try:
            if state in seen:
                continue
            seen.add(state)
        except TypeError as exc:
            note("state is not hashable: %s (%s)" % (_jsonify(state), exc))
            continue
        if len(seen) > caps["max_states"]:
            caps_hit.append("max_states")
            break
        max_depth = max(max_depth, depth)
        label = _jsonify(state)

        if not isinstance(state, frozenset):
            note("state is %s, not a frozenset: %s" % (type(state).__name__, label))
            continue
        try:
            keys = set(dict(state).keys())
        except Exception as exc:
            note("state is not a frozenset of (name, value) pairs: %s (%s)" % (label, exc))
            continue
        if keys != declared_vars:
            note("state variables disagree with the declared catalogue: missing=%s undeclared=%s in %s"
                 % (sorted(declared_vars - keys), sorted(keys - declared_vars), label))

        try:
            terminal = game.is_terminal(state)
        except Exception as exc:
            note("is_terminal raised %s in %s: %s" % (type(exc).__name__, label, exc))
            continue
        if terminal:
            if len(terminals) < caps["max_samples"]:
                terminals.append(label)
            continue

        legal = {}
        for p in players:
            try:
                acts = game.legal_actions(state, p)
            except Exception as exc:
                note("legal_actions raised %s for player '%s' in %s: %s"
                     % (type(exc).__name__, p, label, exc))
                acts = []
            if not acts:
                note("player '%s' has no legal actions in non-terminal state %s" % (p, label))
            for a in acts:
                if a not in catalogue:
                    note("action '%s' for player '%s' is not in the action catalogue" % (a, p))
            actions_used.update(acts)
            legal[p] = list(acts)
        if any(not v for v in legal.values()):
            continue

        profiles = list(itertools.product(*(legal[p] for p in players)))
        if len(profiles) > caps["max_profiles"]:
            caps_hit.append("max_profiles")
            profiles = profiles[:caps["max_profiles"]]

        for profile in profiles:
            actions = dict(zip(players, profile))
            try:
                dist = game.transition_kernel(state, actions)
            except Exception as exc:
                note("transition_kernel raised %s in %s under %s: %s"
                     % (type(exc).__name__, label, actions, exc))
                continue
            if not isinstance(dist, dict) or not dist:
                note("transition_kernel returned an empty/invalid distribution in %s under %s"
                     % (label, actions))
                continue

            total = 0.0
            for key, prob in dist.items():
                n_entries += 1
                if not (isinstance(key, tuple) and len(key) == 2):
                    note("kernel key is not a (next_state, observation_profile) pair: %s" % _jsonify(key))
                    continue
                next_state, obs_profile = key
                if not isinstance(prob, (int, float)) or isinstance(prob, bool):
                    note("probability %r is not a number in %s under %s" % (prob, label, actions))
                    continue
                if prob < 0 or prob > 1:
                    note("probability %s outside [0, 1] in %s under %s" % (prob, label, actions))
                total += float(prob)
                probabilities.add(round(float(prob), 6))

                try:
                    obs = dict(obs_profile)
                except Exception:
                    note("observation profile is not a frozenset of (player, observation) pairs: %s"
                         % _jsonify(obs_profile))
                    obs = {}
                if set(obs.keys()) != set(players):
                    note("observation profile players %s != declared players %s"
                         % (sorted(map(str, obs.keys())), sorted(players)))
                for p, o in obs.items():
                    bucket = observations.setdefault(str(p), set())
                    if len(bucket) < caps["max_samples"]:
                        bucket.add(repr(o))

                try:
                    rew = game.rewards(state, actions, next_state)
                except Exception as exc:
                    note("rewards raised %s in %s under %s: %s"
                         % (type(exc).__name__, label, actions, exc))
                    rew = None
                if rew is not None:
                    if not isinstance(rew, dict) or set(rew.keys()) != set(players):
                        note("rewards keys %s != declared players %s"
                             % (sorted(map(str, rew)) if isinstance(rew, dict) else repr(rew),
                                sorted(players)))
                    else:
                        for p, r in rew.items():
                            if not isinstance(r, (int, float)) or isinstance(r, bool):
                                note("reward %r for player '%s' is not a number" % (r, p))
                        if len(reward_samples) < caps["max_samples"]:
                            reward_samples.append({"actions": _jsonify(actions),
                                                   "rewards": _jsonify(rew)})

                if next_state not in seen:
                    if len(seen) + len(frontier) < caps["max_states"]:
                        frontier.append((next_state, depth + 1))
                    else:                      # truncated: report it, never hide it
                        caps_hit.append("max_states")

            if not math.isclose(total, 1.0, abs_tol=1e-5):
                note("probabilities sum to %.6f (expected 1.0) in %s under %s"
                     % (total, label, actions))

        if depth >= caps["max_depth"]:
            caps_hit.append("max_depth")
            frontier = [(s, d) for s, d in frontier if d <= caps["max_depth"]]

    if not terminals:
        warnings.append("no terminal state was reached within the exploration caps")
    unused = sorted(catalogue - actions_used)
    if unused:
        warnings.append("declared in the catalogue but never legal: %s" % unused)

    summary = {"errors": errors, "warnings": warnings, "caps_hit": sorted(set(caps_hit)),
               "n_states_explored": len(seen), "n_terminal_sampled": len(terminals),
               "max_depth": max_depth, "n_transition_entries": n_entries,
               "probabilities": sorted(probabilities)}
    structure = {"terminal_states": terminals,
                 "observations": {p: sorted(v) for p, v in observations.items()},
                 "reward_samples": reward_samples, "actions_used": sorted(actions_used),
                 "unused_catalogue_actions": unused}
    return summary, structure


def main():
    out = {"stage": "start", "ok": False, "error": None, "class_name": None,
           "candidate_classes": [], "abc_compliant": False, "validate_errors": None,
           "sweep": None, "structure": None, "captured_stdout": ""}
    # generated code may print on import; keep stdout clean so the JSON stays parseable
    real_stdout, sys.stdout = sys.stdout, io.StringIO()
    try:
        abc_path, module_path, caps_json = sys.argv[1], sys.argv[2], sys.argv[3]
        caps = json.loads(caps_json)

        out["stage"] = "import_abc"
        AbstractPOSG = _load(abc_path, "posg_abc").AbstractPOSG

        out["stage"] = "import_module"
        mod = _load(module_path, "generated_posg")

        out["stage"] = "find_class"
        candidates = [obj for obj in vars(mod).values()
                      if isinstance(obj, type) and issubclass(obj, AbstractPOSG)
                      and obj is not AbstractPOSG and obj.__module__ == "generated_posg"]
        out["candidate_classes"] = [c.__name__ for c in candidates]
        if not candidates:
            raise TypeError("the module defines no subclass of AbstractPOSG")
        cls = candidates[0]
        out["class_name"] = cls.__name__

        out["stage"] = "instantiate"
        game = cls()
        out["abc_compliant"] = True

        out["stage"] = "validate"
        try:
            out["validate_errors"] = list(game.validate())
        except Exception as exc:
            # validate() propagates exceptions raised by is_terminal/legal_actions; keep going
            # so the sweep can still say *where* the implementation breaks.
            out["validate_errors"] = ["validate() raised %s: %s" % (type(exc).__name__, exc)]

        out["stage"] = "sweep"
        summary, structure = sweep(game, caps)
        out["sweep"] = summary

        out["stage"] = "structure"

        def _initial_actions(p):
            try:
                return list(game.legal_actions(game.initial_state, p))
            except Exception as exc:
                return ["<error: %s: %s>" % (type(exc).__name__, exc)]

        structure.update({
            "players": {str(p): (m.get("description") if isinstance(m, dict) else _jsonify(m))
                        for p, m in game.players.items()},
            "state_variables": {str(v): _jsonify(m) for v, m in game.state_variables.items()},
            "action_catalogue": {str(a): (m.get("description") if isinstance(m, dict) else _jsonify(m))
                                 for a, m in game.action_catalogue.items()},
            "initial_state": _jsonify(game.initial_state),
            "legal_actions_initial": {str(p): _initial_actions(p) for p in game.players},
        })
        out["structure"] = structure
        out["stage"] = "done"
        out["ok"] = True
    except BaseException as exc:
        out["error"] = _err(exc)
    finally:
        out["captured_stdout"] = sys.stdout.getvalue()[-2000:]
        sys.stdout = real_stdout
    print(json.dumps(out, default=str))


if __name__ == "__main__":
    main()
'''

PROBE_PATH = RUN_DIR / "probe.py"
PROBE_PATH.write_text(PROBE_SOURCE)

MODULE_PREAMBLE = ("# generated by python_prompting_situational_tests.ipynb\n"
                   "from posg_abc import AbstractPOSG, freeze, unfreeze\n"
                   "from typing import Any, Dict, FrozenSet, List, Optional, Tuple\n"
                   "import itertools, math, random\n\n")


def write_module(code: str, path: Path) -> Path:
    path.write_text(MODULE_PREAMBLE + code.rstrip() + "\n")
    return path


def run_probe(module_path: Path) -> dict:
    """Import, instantiate, validate and sweep the module in a fresh interpreter."""
    try:
        proc = subprocess.run(
            [sys.executable, str(PROBE_PATH), str(ABC_PATH), str(module_path),
             json.dumps(CFG.caps)],
            capture_output=True, text=True, timeout=CFG.probe_timeout_s, cwd=str(RUN_DIR),
            env={**os.environ, "PYTHONDONTWRITEBYTECODE": "1"})
    except subprocess.TimeoutExpired:
        return {"stage": "timeout", "ok": False, "error": {
            "type": "Timeout", "message": f"probe exceeded {CFG.probe_timeout_s}s "
            "(runaway state space or non-terminating code)", "traceback": ""}}
    try:
        return json.loads(proc.stdout)
    except json.JSONDecodeError:
        return {"stage": "probe_crash", "ok": False, "error": {
            "type": "ProbeCrash", "message": (proc.stderr or proc.stdout)[-2000:], "traceback": ""}}


def mechanical_verdict(code: str, probe: dict) -> dict:
    """Flatten probe output into the levels reported in the results table."""
    syntax_ok, syntax_error = True, ""
    if code:
        try:
            ast.parse(code)
        except SyntaxError as exc:
            syntax_ok, syntax_error = False, f"SyntaxError: {exc}"
    else:
        syntax_ok, syntax_error = False, "no code in the response"

    stage = probe.get("stage")
    err = probe.get("error") or {}
    validate_errors = probe.get("validate_errors") or []
    sweep = probe.get("sweep") or {}
    sweep_errors = sweep.get("errors") or []

    import_ok = stage not in ("import_module", "import_abc", "probe_crash", "timeout") and syntax_ok
    class_ok = bool(probe.get("class_name"))
    abc_ok = bool(probe.get("abc_compliant"))
    validate_ok = abc_ok and probe.get("validate_errors") is not None and not validate_errors
    sweep_ok = probe.get("sweep") is not None and not sweep_errors
    verdict = {
        "syntax_ok": syntax_ok, "import_ok": import_ok, "class_found": class_ok,
        "abc_ok": abc_ok, "validate_ok": validate_ok, "sweep_ok": sweep_ok,
        "mechanical_ok": bool(syntax_ok and import_ok and class_ok and abc_ok
                              and validate_ok and sweep_ok),
        "failed_stage": None, "errors": [],
    }
    if not syntax_ok:
        verdict["failed_stage"], verdict["errors"] = "syntax", [syntax_error]
    elif not import_ok or not class_ok or not abc_ok:
        verdict["failed_stage"] = stage
        verdict["errors"] = [f"{err.get('type', 'Error')}: {err.get('message', '')}".strip()]
    elif validate_errors:
        verdict["failed_stage"], verdict["errors"] = "validate", validate_errors
    elif sweep_errors:
        verdict["failed_stage"], verdict["errors"] = "sweep", sweep_errors
    elif err:
        verdict["failed_stage"] = stage
        verdict["errors"] = [f"{err.get('type', 'Error')}: {err.get('message', '')}".strip()]
        verdict["mechanical_ok"] = False
    return verdict


### F.1 Harness self-test (runs without an API key)

Before any generation, the two reference implementations from the interface notebook are put
through the harness, together with a copy of Rock-Paper-Scissors mutated to make its
probabilities sum to 0.9. A harness that passes the references and catches the mutation is
working; this is the same negative-testing idea as the 17-mutation self-test in the JSON
notebook's appendix, kept deliberately small.


In [ ]:
def _reference_implementations() -> dict:
    nb = json.loads(POSG_NOTEBOOK.read_text())
    cells = ["".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code"]
    pick = lambda needle: next(s for s in cells if needle in s)
    return {"reference_rps": pick("class RockPaperScissorsPOSG"),
            "reference_dice": pick("class DiceDuelPOSG")}


_refs = _reference_implementations()
_refs["mutated_rps_sum_0.9"] = _refs["reference_rps"].replace(
    "return {(next_state, obs_profile): 1.0}", "return {(next_state, obs_profile): 0.9}")
assert _refs["mutated_rps_sum_0.9"] != _refs["reference_rps"], \
    "the mutation matched nothing — the reference implementation changed, fix the self-test"

_selftest = []
for _name, _src in _refs.items():
    _probe = run_probe(write_module(_src, RUN_DIR / "generated" / f"_selftest_{_name}.py"))
    _v = mechanical_verdict(_src, _probe)
    _selftest.append({"fixture": _name, "mechanical_ok": _v["mechanical_ok"],
                      "failed_stage": _v["failed_stage"],
                      "validate_errors": len(_probe.get("validate_errors") or []),
                      "sweep_errors": len((_probe.get("sweep") or {}).get("errors") or []),
                      "first_error": (_v["errors"] or [""])[0][:90]})

_selftest_df = pd.DataFrame(_selftest)
_expected = {"reference_rps": True, "reference_dice": True, "mutated_rps_sum_0.9": False}
_ok = all(r["mechanical_ok"] == _expected[r["fixture"]] for r in _selftest)
print("harness self-test:", "PASSED" if _ok else "FAILED — do not trust the results below")
_selftest_df


## G. Source audit, one attempt, and the repair loop

The **source audit** is a static (`ast`) read of the generated code, not a judgement. It
reports where numbers came from — declared `# UNSUPPORTED:` constants versus bare numeric
literals inside `transition_kernel` — so that arbitrary invented probabilities are visible
in the results rather than buried in the code.

The **repair loop** feeds the exact validator/sweep errors back and asks for the smallest
correction. Every attempt is kept, so the pass rate before repair, the pass rate after
repair, the number of repair rounds and the extra token cost can all be read off the results
table. Repair is capped (`Config.max_repairs`, default 1): the point of a baseline is to make
failures observable, not to iterate until they disappear.


In [ ]:
def source_audit(code: str) -> dict:
    """Static read of the generated source. Reports provenance of numbers, never a verdict."""
    audit = {"unsupported_constants": [], "bare_probability_literals": [],
             "assumption_comments": [], "n_lines": len(code.splitlines()) if code else 0,
             "parse_error": None}
    if not code:
        return audit
    for line in code.splitlines():
        if re.search(r"#.*\b(UNSUPPORTED|ASSUMPTION|ASSUMED)\b", line, re.I):
            audit["assumption_comments"].append(line.strip()[:160])
    try:
        tree = ast.parse(code)
    except SyntaxError as exc:
        audit["parse_error"] = str(exc)
        return audit

    named = {}
    for node in tree.body:
        if isinstance(node, ast.Assign):
            for target in node.targets:
                if isinstance(target, ast.Name):
                    named[target.id] = ast.unparse(node.value)
    commented = " ".join(audit["assumption_comments"])
    audit["unsupported_constants"] = [f"{k} = {v}" for k, v in named.items()
                                      if k in commented or k.upper().startswith("P_")]

    for node in ast.walk(tree):
        if isinstance(node, ast.FunctionDef) and node.name == "transition_kernel":
            for sub in ast.walk(node):
                if (isinstance(sub, ast.Constant) and isinstance(sub.value, float)
                        and 0.0 < sub.value < 1.0):
                    audit["bare_probability_literals"].append(sub.value)
    return audit


def run_attempt(case: dict, gen: int, repair: int, previous_code: str = "",
                errors: str = "", stage: str = "") -> dict:
    """One LLM call plus the full mechanical pipeline. Everything is saved to RUN_DIR."""
    tag = f"{case['case_id'].replace('.', '')}_g{gen}_r{repair}"
    raw, code, meta, prompts = generate(case, previous_code, errors, stage)

    (RUN_DIR / "prompts" / f"{tag}.md").write_text(
        f"# SYSTEM\n\n{prompts['system']}\n\n# USER\n\n{prompts['user']}\n")
    (RUN_DIR / "raw" / f"{tag}.txt").write_text(raw or "")

    module_path = None
    probe = {"stage": "no_code", "ok": False,
             "error": {"type": "NoCode", "message": "no code in the response", "traceback": ""}}
    if code:
        module_path = write_module(code, RUN_DIR / "generated" / f"{tag}.py")
        probe = run_probe(module_path)
    (RUN_DIR / "probe" / f"{tag}.json").write_text(json.dumps(probe, indent=2, default=str))

    verdict = mechanical_verdict(code, probe)
    if not code and meta.get("api_error"):        # the call itself failed
        verdict["failed_stage"], verdict["errors"] = "api_error", [meta["api_error"]]
    elif not code and meta.get("refusal"):        # the model declined
        verdict["failed_stage"] = "refusal"
        verdict["errors"] = [f"refusal ({meta.get('refusal_category')})"]
    return {"case_id": case["case_id"], "title": case["title"], "generation": gen,
            "repair_round": repair, "tag": tag, "code_generated": bool(code),
            "code": code, "meta": meta, "probe": probe, "verdict": verdict,
            "audit": source_audit(code),
            "module_path": str(module_path.relative_to(RUN_DIR)) if module_path else None}


def run_case(case: dict) -> list:
    """All generations for one case, each with up to CFG.max_repairs repair rounds."""
    attempts = []
    for gen in range(CFG.n_generations):
        attempt = run_attempt(case, gen, 0)
        attempts.append(attempt)
        for repair in range(1, CFG.max_repairs + 1):
            if attempt["verdict"]["mechanical_ok"] or not attempt["code_generated"]:
                break
            attempt = run_attempt(case, gen, repair,
                                  previous_code=attempt["code"],
                                  errors="\n".join(attempt["verdict"]["errors"]),
                                  stage=attempt["verdict"]["failed_stage"] or "unknown")
            attempts.append(attempt)
    return attempts


## H. Run the experiment

One row per attempt is printed as it completes. In `dry_run` mode this exercises the whole
pipeline against the stub — including one deliberately broken generation (case S.5) so the
repair path runs.


In [ ]:
ATTEMPTS = []
for _case in CASES:
    for _a in run_case(_case):
        ATTEMPTS.append(_a)
        _v = _a["verdict"]
        print(f"{_a['tag']:>14}  code={'y' if _a['code_generated'] else 'n'}  "
              f"mech={'PASS' if _v['mechanical_ok'] else 'FAIL':4s}  "
              f"stage={str(_v['failed_stage'] or '-'):9s}  "
              f"tok_in={_a['meta'].get('input_tokens')} tok_out={_a['meta'].get('output_tokens')}  "
              f"{(_v['errors'] or [''])[0][:70]}")
print(f"\n{len(ATTEMPTS)} attempts over {len(CASES)} cases")


## I. Results table

`attempts_df` has one row per LLM call; `results_df` has one row per (case, generation) and
is the headline table: what happened before repair, after repair, at what token cost, plus
the mechanical flags that section J's review sheets expand on.

The flags are **pointers for a human reviewer, not scores**: for example
`flag_no_uncertainty` says every probability in the model is 0 or 1, which *may* mean a live
uncertainty was collapsed — the reviewer decides.


In [ ]:
FLAG_KEYS = ("n_players", "n_state_vars", "n_actions", "n_states_explored", "max_depth",
             "n_terminal_sampled", "caps_hit", "probabilities", "flag_two_players_only",
             "flag_no_uncertainty", "flag_bare_probability_literals",
             "flag_declared_unsupported", "flag_single_step", "flag_no_terminal_reached",
             "flag_identical_observations", "flag_unused_actions")


def flags(attempt: dict) -> dict:
    """Mechanical signals that a reviewer should look at. Not judgements."""
    st = (attempt["probe"] or {}).get("structure") or {}
    sw = (attempt["probe"] or {}).get("sweep") or {}
    audit = attempt["audit"]
    if not st:                       # nothing ran: report blanks rather than misleading zeros
        return {k: (False if k.startswith("flag_") else None) for k in FLAG_KEYS}
    players = st.get("players") or {}
    probs = sw.get("probabilities") or []
    obs = st.get("observations") or {}
    non_trivial = [p for p in probs if 0.0 < p < 1.0]
    return {
        "n_players": len(players),
        "n_state_vars": len(st.get("state_variables") or {}),
        "n_actions": len(st.get("action_catalogue") or {}),
        "n_states_explored": sw.get("n_states_explored"),
        "max_depth": sw.get("max_depth"),
        "n_terminal_sampled": sw.get("n_terminal_sampled"),
        "caps_hit": ",".join(sw.get("caps_hit") or []),
        "probabilities": non_trivial,
        "flag_two_players_only": len(players) <= 2,
        "flag_no_uncertainty": bool(probs) and not non_trivial,
        "flag_bare_probability_literals": len(audit["bare_probability_literals"]),
        "flag_declared_unsupported": len(audit["unsupported_constants"]),
        "flag_single_step": (sw.get("max_depth") or 0) <= 1,
        "flag_no_terminal_reached": sw.get("n_terminal_sampled") == 0,
        "flag_identical_observations": (len(obs) > 1 and
                                        len({tuple(v) for v in obs.values()}) == 1),
        "flag_unused_actions": len(st.get("unused_catalogue_actions") or []),
    }


attempts_rows = []
for _a in ATTEMPTS:
    _v, _m = _a["verdict"], _a["meta"]
    attempts_rows.append({
        "case_id": _a["case_id"], "generation": _a["generation"], "repair_round": _a["repair_round"],
        "model": CFG.model if not CFG.dry_run else "dry-run-stub",
        "served_by": _m.get("served_by"), "stop_reason": _m.get("stop_reason"),
        "code_generated": _a["code_generated"], "syntax_ok": _v["syntax_ok"],
        "import_ok": _v["import_ok"], "abc_ok": _v["abc_ok"], "validate_ok": _v["validate_ok"],
        "sweep_ok": _v["sweep_ok"], "mechanical_ok": _v["mechanical_ok"],
        "failed_stage": _v["failed_stage"],
        "first_error": (_v["errors"] or [""])[0][:300],
        "n_errors": len(_v["errors"]),
        "input_tokens": _m.get("input_tokens"), "output_tokens": _m.get("output_tokens"),
        "api_error": _m.get("api_error"),
        "class_name": (_a["probe"] or {}).get("class_name"),
        "module_path": _a["module_path"], "tag": _a["tag"], **flags(_a)})
attempts_df = pd.DataFrame(attempts_rows)


def summarise(case_id: str, gen: int) -> dict:
    """One row per (case, generation): the initial attempt, the repairs, the final state."""
    rows = [a for a in ATTEMPTS if a["case_id"] == case_id and a["generation"] == gen]
    first, last = rows[0], rows[-1]
    repairs = [r for r in rows if r["repair_round"] > 0]
    case = next(c for c in CASES if c["case_id"] == case_id)
    return {
        "case_id": case_id, "generation": gen, "title": first["title"],
        "situation": case["situation"][:120] + ("..." if len(case["situation"]) > 120 else ""),
        "model": CFG.model if not CFG.dry_run else "dry-run-stub",
        "code_generated": first["code_generated"],
        "syntax_ok": first["verdict"]["syntax_ok"],
        "import_ok": first["verdict"]["import_ok"],
        "abc_ok": first["verdict"]["abc_ok"],
        "validate_ok_before_repair": first["verdict"]["validate_ok"],
        "mechanical_ok_before_repair": first["verdict"]["mechanical_ok"],
        "validation_error": (first["verdict"]["errors"] or [""])[0][:200],
        "repair_attempted": bool(repairs),
        "n_repair_attempts": len(repairs),
        "mechanical_ok_after_repair": last["verdict"]["mechanical_ok"],
        "input_tokens_total": sum(r["meta"].get("input_tokens") or 0 for r in rows),
        "output_tokens_total": sum(r["meta"].get("output_tokens") or 0 for r in rows),
        "repair_output_tokens": sum(r["meta"].get("output_tokens") or 0 for r in repairs),
        "class_name": (last["probe"] or {}).get("class_name"),
        "module_path": last["module_path"],
        "semantic_review": "pending_human_review" if last["code_generated"] else "n/a_no_model",
        **flags(last),
    }


results_df = pd.DataFrame([summarise(c["case_id"], g)
                           for c in CASES for g in range(CFG.n_generations)])

attempts_df.to_csv(RUN_DIR / "attempts.csv", index=False)
results_df.to_csv(RUN_DIR / "results.csv", index=False)
(RUN_DIR / "records.json").write_text(json.dumps(
    [{k: v for k, v in a.items() if k != "code"} for a in ATTEMPTS], indent=2, default=str))

_headline = ["case_id", "generation", "code_generated", "abc_ok", "mechanical_ok_before_repair",
             "n_repair_attempts", "mechanical_ok_after_repair", "output_tokens_total",
             "n_players", "max_depth", "semantic_review"]
_n = len(results_df)
print(f"mechanically valid before repair: {results_df['mechanical_ok_before_repair'].sum()}/{_n}"
      f"  (case x generation)")
print(f"mechanically valid after  repair: {results_df['mechanical_ok_after_repair'].sum()}/{_n}")
results_df[_headline]


In [ ]:
# where the first attempt failed, and on what
if not attempts_df.empty:
    display(attempts_df[attempts_df.repair_round == 0]
            .groupby("failed_stage", dropna=False).size().rename("n_cases").to_frame())
    display(attempts_df.loc[~attempts_df.mechanical_ok,
                            ["case_id", "repair_round", "failed_stage", "first_error"]])


## J. Semantic inspection — human review, not an automated score

Lane 3 of the evaluation methodology (expect/avoid ingredient coverage, bias checks, human
review) **is not implemented in this repository**, and this notebook does not fabricate one.
What it does instead is prepare the review: for every case it writes a Markdown sheet that
puts the extracted model structure next to the case's own *should include* / *should NOT
include* bullets, with checkboxes for a reviewer.

Each sheet contains exactly the structured material needed to compare a Python model against
those checklists later — players, state variables, actions, initial legal actions, terminal
states, per-player observations, the probabilities actually used, and the provenance of
those numbers. This is the input a future automated Lane 3 would need too.

**Watch for, in particular:** an invented stakeholder or a missing low-power one; an
uncertainty collapsed into an assumption; a player observing a latent variable they should
not see, or failing to observe something about themselves; an action that presumes authority
the robot does not have; arbitrary numbers presented as if known; and models that are far
larger than the situation needs.


In [ ]:
def review_sheet(case: dict, attempt: dict) -> str:
    st = (attempt["probe"] or {}).get("structure") or {}
    sw = (attempt["probe"] or {}).get("sweep") or {}
    v, audit, f = attempt["verdict"], attempt["audit"], flags(attempt)
    L = [f"# {case['case_id']} — {case['title']}", "",
         f"*Model:* `{CFG.model if not CFG.dry_run else 'dry-run-stub'}` · "
         f"*attempt:* `{attempt['tag']}` · "
         f"*mechanical:* **{'PASS' if v['mechanical_ok'] else 'FAIL'}**"
         + ("" if v["mechanical_ok"] else f" (at `{v['failed_stage']}`)"),
         f"*Generated module:* `{attempt['module_path']}`", "",
         "## Situation (verbatim from the repository)", "", f"> {case['situation']}", ""]

    if not attempt["code_generated"]:
        L += ["## No model was produced", "", f"`{(v['errors'] or [''])[0]}`", ""]
    else:
        L += ["## Players", ""]
        L += [f"- `{p}` — {d}" for p, d in (st.get("players") or {}).items()] or ["- (none)"]
        L += ["", "## State variables", ""]
        for name, meta in (st.get("state_variables") or {}).items():
            desc = meta.get("description", "") if isinstance(meta, dict) else meta
            L.append(f"- `{name}` — {desc}")
        L += ["", f"Initial state: `{st.get('initial_state')}`", "",
              "## Actions", ""]
        for a, d in (st.get("action_catalogue") or {}).items():
            L.append(f"- `{a}` — {d}")
        L += ["", "Legal at the initial state:", ""]
        for p, acts in (st.get("legal_actions_initial") or {}).items():
            L.append(f"- `{p}`: {acts}")
        L += ["", "## Terminal states reached", ""]
        L += [f"- `{t}`" for t in (st.get("terminal_states") or [])] or ["- none reached"]
        L += ["", "## Observations seen, per player", ""]
        for p, obs in (st.get("observations") or {}).items():
            L.append(f"- `{p}`: {obs}")
        L += ["", "## Numbers and their provenance", "",
              f"- probabilities used: `{sw.get('probabilities')}`",
              f"- declared `# UNSUPPORTED` constants: `{audit['unsupported_constants'] or 'none'}`",
              f"- bare probability literals inside `transition_kernel`: "
              f"`{audit['bare_probability_literals'] or 'none'}`",
              f"- assumption comments: `{audit['assumption_comments'] or 'none'}`", "",
              "## Size", "",
              f"- {f['n_players']} players · {f['n_state_vars']} state variables · "
              f"{f['n_actions']} actions",
              f"- {f['n_states_explored']} states explored, depth {f['max_depth']}, "
              f"{sw.get('n_transition_entries')} transition entries"
              + (f", caps hit: {f['caps_hit']}" if f["caps_hit"] else ""), "",
              "## Mechanical flags (pointers, not verdicts)", ""]
        L += [f"- `{k}` = `{val}`" for k, val in f.items() if k.startswith("flag_") and val]
        L += ["- none raised"] if not any(v2 for k, v2 in f.items() if k.startswith("flag_")) else []

    L += ["", "## Reviewer checklist — the model **should include**", ""]
    L += [f"- [ ] {item}" for item in case["expect"]]
    L += ["", "## Reviewer checklist — the model **should NOT include**", ""]
    L += [f"- [ ] {item}" for item in case["avoid"]]
    L += ["", "## Reviewer notes", "", "_(free text: omissions, inventions, collapsed "
          "uncertainties, observation errors, over-modelling)_", ""]
    return "\n".join(L)


def last_attempt(case_id: str, gen: int) -> dict:
    return [a for a in ATTEMPTS if a["case_id"] == case_id and a["generation"] == gen][-1]


_n_sheets = 0
for _case in CASES:
    for _gen in range(CFG.n_generations):
        (RUN_DIR / "review" / f"{_case['case_id'].replace('.', '')}_g{_gen}.md").write_text(
            review_sheet(_case, last_attempt(_case["case_id"], _gen)))
        _n_sheets += 1
print(f"wrote {_n_sheets} review sheets to {RUN_DIR / 'review'}")


In [ ]:
display(Markdown(review_sheet(CASES[0], last_attempt(CASES[0]["case_id"], 0))))


## K. Outputs, assumptions, and open questions

### Saved artefacts (everything under the run directory)

| Path | Contents |
|---|---|
| `config.json` | the exact configuration of this run |
| `posg_abc.py` | interface cells extracted from the reference notebook (derived, do not edit) |
| `probe.py` | the subprocess validator/sweep |
| `prompts/*.md` | the exact system and user prompt of every call |
| `raw/*.txt` | raw model responses |
| `generated/*.py` | executable modules (preamble + generated class) |
| `probe/*.json` | full probe output per attempt |
| `attempts.csv`, `results.csv`, `records.json` | the results tables |
| `review/*.md` | one human-review sheet per case and generation |

Run directories are experiment artefacts, not source: commit one deliberately when it is
worth keeping, or add `experiments/` to `.gitignore`. On Colab the repository checkout is
ephemeral — copy the run directory to Drive before the session ends. Nothing outside the run
directory is written; the interface notebook and the test-case files are only read.

### What this notebook reuses, and what it adds

**Reused:** `AbstractPOSG`, `freeze`/`unfreeze` and `validate()` from
`notebooks/POSG_World_Model_Implementation.ipynb` (extracted, never copied); the two
reference implementations, as harness fixtures; the eleven situational cases and their
expect/avoid checklists from `data/Test_Cases/02_situational/`; the subprocess-isolation
pattern from `llm_world_model_experiment/run_experiment.py`; the "dynamics only" framing and
the given/inferred/invented discipline from `prompts_library/zero_shot_prompt.md` and
`docs/representation_science/`.

**Added here:** the Python generation prompt; the model-calling seam; code extraction; the
joint-action sweep and the extra mechanical checks (state-variable agreement, reward vectors,
observation coverage, caps); the source audit of numeric provenance; the repair loop; the
results tables; the review sheets.

### Assumptions this notebook makes, which the repository has not settled

1. **Rewards.** The interface makes `rewards` abstract; the JSON prompt says models supply
   dynamics only. The prompt asks for `0.0` per player unless the situation states a payoff.
   Confirm this is what the downstream power analysis wants.
2. **Initial uncertainty.** `initial_state` is a single state, so uncertainty about the
   starting world is carried as a state variable resolved in the first transition. The JSON
   schema instead allows a distribution over initial states — the two tracks therefore differ
   structurally here, which matters for any later cross-track comparison.
3. **Invented probabilities.** The `# UNSUPPORTED:` named-constant convention and the
   source audit are a notebook-level proposal, not ratified methodology.
   `docs/representation_science/open_questions.md` still asks how to separate unsupported
   invention from useful commonsense inference; the audit measures the question, it does not
   answer it.
4. **One class per response.** The probe takes the first `AbstractPOSG` subclass in the
   module; `candidate_classes` in the probe output records if there were several.
5. **Lane 2 does not apply.** The `game_matching` structural comparison needs a reference
   model, and the situational cases deliberately have none.

### What the baseline is for

Read the failure modes, not the pass rate: which mechanical errors recur (they are prompt
bugs as much as model bugs), whether one repair round clears them and at what token cost, and
which recurring semantic failures the review sheets expose. Those observations, not a score,
are what the next prompt iteration should be built on.


In [ ]:
print("run directory:", RUN_DIR)
for _p in sorted(RUN_DIR.rglob("*")):
    if _p.is_file():
        print(f"  {_p.relative_to(RUN_DIR)}  ({_p.stat().st_size} bytes)")
